# Feature Engineering Notebook
## AmEx Offer Ranking — Learning to Rank (LTR)

This notebook transforms raw train/test CSVs into a rich feature set for the offer ranking task.

### Feature Groups Built

| Group | Description |
|---|---|
| **Smart null handling** | Binary null-indicator flags + group-median imputation for features with 14–17% missingness (f130, f131, f133, f138, f362–f366) |
| **OHE reversal** | One-hot encoded category (f226–f232) and subcategory (f233–f309) columns collapsed into single integer-coded columns `offer_category` / `offer_subcategory` + multi-label count columns |
| **Sequential / session features** | `time_since_last_seen`, `session_event_count`, `previous_offer_category`, `no_of_clicks_previously`, `time_since_last_click` — all via `.shift(1)` so no future leakage |
| **Customer-level aggregations** | Interest scores (f1–f12), digital channel counts (f23–f27), spend (f152–f173), loyalty/miles (f43, f45, f58), non-merchant CTR (f104–f111), transaction diversity and decay signals |
| **Offer-level features** | CTR momentum (1d/7d ratio), value thresholds, age/decay acceleration, merchant CTR momentum |
| **Customer × Offer interactions** | ~30 cross features: interest × category CTR, spend × CTR, session depth × offer value, industry CTR alignment, merchant engagement |
| **Composite relevance score** | Weighted blend of top-4 correlated features (f366 w=0.40, f363 w=0.30, f365 w=0.20, f362 w=0.10) |
| **Within-group rank features** | **REMOVED** — confirmed data leakage (used full-group statistics that leak future rows) |

**Output files:** `featured_full_train.csv` / `featured_full_test.csv`

---
### Leakage Controls
- Categorical encoding fitted on **train rows only** (`_is_train == True`)
- Sequential features use `.shift(1)` — current row never sees its own or future data
- Binary value thresholds derived from **train distribution only**
- Within-group rank/zscore features removed after leakage confirmation

In [92]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Load Data

Reads the raw `full_train.csv` and `full_test.csv` files, then concatenates them into a single `df` for joint feature engineering. A boolean `_is_train` flag marks each row's origin so we can split back at the end.

Helper functions `safe_div(a, b)` (zero-safe division) and `col_exists` / `existing_cols` (column presence checks) are also defined here.

In [93]:
train_df = pd.read_csv('/Users/akashjaiswal/Desktop/amex-hackathon-4/Data/full_train.csv')
test_df = pd.read_csv('/Users/akashjaiswal/Desktop/amex-hackathon-4/Data/full_test.csv')

print(f'Train shape: {train_df.shape}')
print(f'Test shape : {test_df.shape}')

Train shape: (616602, 316)
Test shape : (153544, 316)


In [94]:
# Mark source so we can split later
train_df['_is_train'] = True
test_df['_is_train'] = False

# If test has no 'y' column, add a placeholder
if 'y' not in test_df.columns:
    test_df['y'] = np.nan

# Concatenate train + test for unified feature engineering
df = pd.concat([train_df, test_df], ignore_index=True)

# Parse id4 as datetime
df['id4'] = pd.to_datetime(df['id4'])

print(f'Combined   : {df.shape}')
print(f'\nTarget distribution (train only):')
print(train_df['y'].value_counts())
print(f'\nOffers per customer (median): {df.groupby("id2")["y"].count().median()}')
df.head(3)

Combined   : (770146, 317)

Target distribution (train only):
y
0    586535
1     30067
Name: count, dtype: int64

Offers per customer (median): 3.0


,id1,id2,id3,id4,id5,y,f1,f2,f5,f6,f7,f8,f9,f10,f11,f12,f22,f23,f24,f25,f26,f27,f28,f30,f31,f32,f38,f39,f41,f42,f43,f44,f45,f46,f47,f48,f49,f50,f51,f52,f53,f54,f55,f56,f57,f58,f59,f60,f61,f62,...,f317,f318,f319,f320,f321,f322,f323,f324,f325,f326,f327,f328,f329,f330,f331,f332,f333,f334,f335,f336,f337,f338,f339,f340,f341,f342,f343,f344,f345,f346,f347,f348,f349,f350,f351,f352,f353,f354,f355,f356,f357,f358,f359,f361,f362,f363,f364,f365,f366,_is_train
0,1366776_189706075_16-23_2023-11-02 22:22:00.042,1366776,189706075,2023-11-02 22:22:00.042,2023-11-02,0,1.0000,NaN,NaN,NaN,NaN,NaN,NaN,13.0000,27.0000,NaN,2.0000,0.0000,0.0000,0.0000,2.0000,0.0000,27.1294,418.2614,4.8689,NaN,16.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1114698.0000,0.0000,0.0000,0.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0000,0.0000,0.0000,0.0000,0.1030,0.1400,0.1140,0.0953,0.0800,0.7357,1.2282,1.1964,244278.0000,1388862.0000,1388862.0000,NaN,NaN,5,80458.0000,1.0000,3.0000,0.0000,Phase_1,NaN,NaN,NaN,-9999.0000,0.0000,28.0000,0.0000,0.0000,337.0000,0.0000,0.0000,True
1,1366776_89227_16-23_2023-11-01 23:51:24.999,1366776,89227,2023-11-01 23:51:24.999,2023-11-01,0,1.0000,NaN,NaN,NaN,NaN,NaN,NaN,13.0000,27.0000,NaN,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,29.0766,425.5725,5.2184,NaN,18.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1114698.0000,0.0000,0.0000,0.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0000,0.0000,0.0000,0.0000,0.0995,0.0979,0.0880,0.0796,0.0707,1.0162,1.1123,1.1056,331455.0000,2210070.0000,12679035.0000,NaN,NaN,4,85874.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,87.0000,0.0000,0.0000,1010.0000,2.0000,0.0020,True
2,1366776_35046_16-23_2023-11-01 00:30:59.797,1366776,35046,2023-11-01 00:30:59.797,2023-11-01,0,1.0000,NaN,NaN,NaN,NaN,NaN,NaN,13.0000,27.0000,NaN,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,15.0869,223.4106,1.3058,NaN,18.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,986.0000,0.0000,0.0000,0.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0000,0.0000,0.0000,0.0000,0.0839,0.1221,0.1035,0.0820,0.0687,0.6872,1.1797,1.2622,242568.0000,1323567.0000,1323567.0000,NaN,NaN,4,1855.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,23.0000,0.0000,0.0000,1010.0000,2.0000,0.0020,True


In [95]:
# ── Helpers ──────────────────────────────────────────────────────────
def safe_div(a, b, fill=0.0):
    """Divide avoiding division by zero."""
    return np.where(b == 0, fill, a / b)

def col_exists(df, col):
    return col in df.columns

def existing_cols(df, cols):
    return [c for c in cols if c in df.columns]

def safe_idxmax(df, cols):
    """idxmax that handles all-NaN rows."""
    return df[cols].fillna(0).idxmax(axis=1)

print('Helpers defined.')

Helpers defined.


---
## 2. Encode String Categoricals

String columns that carry categorical meaning (e.g. merchant type, offer channel) are integer-encoded using `pd.CategoricalDtype`. The category codes are **fitted on train rows only** (`_is_train == True`) and then applied to test rows — this prevents test-set distribution from leaking into the encoding and ensures consistent integer codes across splits.

In [96]:
# Fit category codes on train rows ONLY — prevents test distribution leaking into encoding
train_mask = df['_is_train'] == True

str_cat_cols = existing_cols(df, ['f42', 'f50', 'f52', 'f53', 'f54', 'f55', 'f56', 'f57', 'f354'])
for col in str_cat_cols:
    train_vals = df.loc[train_mask, col].astype(str).fillna('__missing__')
    train_cats = sorted(train_vals.unique())
    cat_dtype  = pd.CategoricalDtype(categories=train_cats, ordered=False)
    all_vals   = df[col].astype(str).fillna('__missing__').where(
        df[col].astype(str).fillna('__missing__').isin(train_cats), other='__missing__'
    )
    df[col] = all_vals.astype(cat_dtype).cat.codes  # unknown test categories → same code as '__missing__'

for col in existing_cols(df, ['f48', 'f349']):
    train_vals = df.loc[train_mask, col].astype(str).fillna('__missing__')
    train_cats = sorted(train_vals.unique())
    cat_dtype  = pd.CategoricalDtype(categories=train_cats, ordered=False)
    all_vals   = df[col].astype(str).fillna('__missing__').where(
        df[col].astype(str).fillna('__missing__').isin(train_cats), other='__missing__'
    )
    df[col] = all_vals.astype(cat_dtype).cat.codes

print(f'Encoded {len(str_cat_cols)} string categorical columns (fit on train only, no leakage).')

Encoded 9 string categorical columns (fit on train only, no leakage).


---
## 3. Smart Null Handling on High-Signal Features

**Problem:** Top features f366, f363, f365, f362, f131, f133, f130, f138 have 14–17% nulls. A naive `fillna(0)` makes "no history" look identical to "zero CTR", which corrupts the model's ability to distinguish customers with missing records from customers who genuinely never clicked.

**Fix applied per feature:**
1. Add a binary null-indicator column `<feature>_null` (missingness is itself a predictive signal — it typically means the customer is new or has no history in that category)
2. Fill nulls with the **per-customer group median** (using `groupby(id2).transform('median')`), falling back to the global median for customers with all-null values

In [97]:
high_null_feats = existing_cols(df, [
    'f366', 'f363', 'f365', 'f362',   # customer x industry CTR (top corr: 0.56, 0.50, 0.43, 0.32)
    'f131', 'f133', 'f130', 'f138',   # customer category CTR  (corr: 0.34, 0.33, 0.35, 0.34)
    'f95',  'f94',  'f199', 'f201',   # non-merchant impressions / decayed clicks
])

print('Null rates BEFORE fix:')
for f in high_null_feats:
    print(f'  {f}: {df[f].isnull().mean():.2%}')

train_mask = df['_is_train'] == True
for feat in high_null_feats:
    df[f'{feat}_is_null'] = df[feat].isnull().astype(int)
    # Compute medians on TRAIN rows only — prevents test values from influencing imputation
    global_median = df.loc[train_mask, feat].median()
    group_median  = df.loc[train_mask].groupby('id2')[feat].median()
    df[feat] = df[feat].fillna(df['id2'].map(group_median)).fillna(global_median)

print(f'\nNull indicator flags added: {len(high_null_feats)}')
print('Null rates AFTER fix: all 0% ✓')
print('Imputation medians computed on train rows only (no leakage) ✓')

Null rates BEFORE fix:
  f366: 14.06%
  f363: 15.03%
  f365: 14.06%
  f362: 15.03%
  f131: 15.39%
  f133: 14.17%
  f130: 12.51%
  f138: 9.74%
  f95: 1.00%
  f94: 1.00%
  f199: 0.99%
  f201: 0.99%

Null indicator flags added: 12
Null rates AFTER fix: all 0% ✓
Imputation medians computed on train rows only (no leakage) ✓


---
## 4. Reverse One-Hot Encoding

### 4A. Primary Category (f226–f232)

The raw data one-hot encodes the 7 offer categories (Business, Dining, Entertainment, Retail, Services, Shopping, Travel) across columns f226–f232. This section:

1. **Collapses** the OHE columns into a single integer-coded `offer_category` column using `pd.CategoricalDtype` (fitted on train only)
2. **Computes** `num_offer_categories` — the count of active OHE bits per row, which captures multi-category offers (normally 1, but occasionally > 1)
3. **Drops** the original f226–f232 columns to reduce dimensionality

In [98]:
cat_map = {
    'f226': 'Business', 'f227': 'Dining',   'f228': 'Entertainment',
    'f229': 'Retail',   'f230': 'Services', 'f231': 'Shopping', 'f232': 'Travel'
}
cat_ohe_cols = existing_cols(df, list(cat_map.keys()))
if cat_ohe_cols:
    # num_offer_categories: how many categories this offer spans (v1 insight)
    df['num_offer_categories'] = df[cat_ohe_cols].fillna(0).sum(axis=1).astype(int)

    # Single dominant category via idxmax (v2 approach)
    df['offer_category'] = safe_idxmax(df, cat_ohe_cols).map(cat_map)
    df.loc[df[cat_ohe_cols].fillna(0).sum(axis=1) == 0, 'offer_category'] = 'Unknown'
    df['offer_category'] = df['offer_category'].astype('category').cat.codes

    print(f'offer_category       : {df["offer_category"].nunique()} unique values')
    print(f'num_offer_categories : {df["num_offer_categories"].value_counts().to_dict()}')

offer_category       : 5 unique values
num_offer_categories : {1: 757589, 2: 9544, 4: 3013}


### 4B. Subcategory (f233–f309)

77 subcategory OHE columns (f233–f309) are similarly collapsed into a single `offer_subcategory` integer-coded column. Additional steps:

1. **`num_sub_categories`** — count of active subcategory bits per row; rows where this equals 77 are anomalous (all bits set) and are removed
2. **Drop** all subcategory OHE columns (f233–f309) and the primary category OHE columns (f226–f232) together, leaving only the compact coded columns

In [99]:
subcat_map = {
    'f233': 'Activities_leisure', 'f234': 'Advertising',     'f235': 'American',
    'f236': 'Animal_services',    'f237': 'Apparel',         'f238': 'Argentinean',
    'f239': 'Asian',              'f240': 'Australian',      'f241': 'Auto_fuel',
    'f242': 'Auto_rental',        'f243': 'Auto_sales',      'f244': 'Auto_services',
    'f245': 'Automotive',         'f246': 'Bar_night',       'f247': 'Books_music',
    'f248': 'British',            'f249': 'Carry_out',       'f250': 'Casual',
    'f251': 'Chinese',            'f252': 'Clothing',        'f253': 'Coffee_desserts',
    'f254': 'Communications',     'f255': 'Computer_services','f256': 'Constructions',
    'f257': 'Cruises',            'f258': 'Cuisin',          'f259': 'Delis',
    'f260': 'Education',          'f261': 'Electronics',     'f262': 'European',
    'f263': 'Finance',            'f264': 'Food_store',      'f265': 'Food_wine',
    'f266': 'French',             'f267': 'Fusion',          'f268': 'German',
    'f269': 'Gifts',              'f270': 'Golf',            'f271': 'Greek',
    'f272': 'Health_care',        'f273': 'Hobbies',         'f274': 'Home_furnishing',
    'f275': 'Home_garden',        'f276': 'Hotel_resorts',   'f277': 'Indian',
    'f278': 'Italian',            'f279': 'Japanese',        'f280': 'Jewelry',
    'f281': 'Mediterranean',      'f282': 'Mexican',         'f283': 'Office_products',
    'f284': 'Office_supplies',    'f285': 'Other',           'f286': 'Other_cuisine',
    'f287': 'Other_services',     'f288': 'Other_shopping',  'f289': 'Parking',
    'f290': 'Parks_zoos',         'f291': 'Personal_care',   'f292': 'Printing',
    'f293': 'Professional',       'f294': 'Real_estate',     'f295': 'Residential_care',
    'f296': 'Restaurant',         'f297': 'Seafood',         'f298': 'Services',
    'f299': 'Shipping_services',  'f300': 'Spanish',         'f301': 'Sports_recreation',
    'f302': 'Steakhouse',         'f303': 'Thai',            'f304': 'Tour',
    'f305': 'Transportation',     'f306': 'Travel_agencies', 'f307': 'Utilities',
    'f308': 'Vegetarian',         'f309': 'Wholesale'
}
subcat_ohe_cols = existing_cols(df, list(subcat_map.keys()))

In [100]:
if subcat_ohe_cols:
    # num_sub_categories: count of active bits
    df['num_sub_categories'] = df[subcat_ohe_cols].fillna(0).sum(axis=1).astype(int)

    # Remove anomalous rows where all 77 subcategory bits are set
    n_before = len(df)
    df = df[df['num_sub_categories'] != 77].copy()
    print(f'Removed {n_before - len(df)} anomalous rows (num_sub_categories=77)')

    # Single dominant subcategory via idxmax
    df['offer_subcategory'] = safe_idxmax(df, subcat_ohe_cols).map(subcat_map)
    df.loc[df[subcat_ohe_cols].fillna(0).sum(axis=1) == 0, 'offer_subcategory'] = 'Unknown'
    df['offer_subcategory'] = df['offer_subcategory'].astype('category').cat.codes
    print(f'offer_subcategory    : {df["offer_subcategory"].nunique()} unique values')

Removed 0 anomalous rows (num_sub_categories=77)
offer_subcategory    : 42 unique values


In [101]:
# Drop all OHE columns (categories + subcategories)
ohe_cols_to_drop = existing_cols(df, list(cat_map.keys()) + list(subcat_map.keys()))
df.drop(columns=ohe_cols_to_drop, inplace=True)
print(f'Dropped {len(ohe_cols_to_drop)} OHE columns. Shape: {df.shape}')

Dropped 84 OHE columns. Shape: (770146, 249)


---
## 5. Sequential / Session Features

These features capture the **within-session temporal dynamics** of each customer's offer stream — what happened *before* this offer impression. All are computed after sorting by `[id2, id4]` and use `.shift(1)` so the current row only sees past information:

| Feature | Description | Default (first row) |
|---|---|---|
| `time_since_last_seen` | Seconds elapsed since the previous offer was shown to this customer | `0` |
| `session_event_count` | Cumulative count of offers shown to this customer so far (position in stream) | `0` |
| `previous_offer_category` | Integer-coded category of the immediately prior offer | `-1` (unknown) |
| `previous_suboffer_category` | Subcategory of the immediately prior offer | `-1` |
| `no_of_clicks_previously` | Running total of clicks made by this customer before this row | `0` |
| `time_since_last_click` | Seconds since this customer last clicked any offer | `0` |
| `is_same_category_as_previous` | 1 if current offer shares category with previous offer | `0` |
| `pace_x_ctr` | `time_since_last_seen × f366` — interaction of recency pace with customer CTR | derived |
| `session_count_x_ctr` | `session_event_count × f366` | derived |

In [102]:
# Sort by customer + timestamp — required for correct sequential feature computation
df = df.sort_values(['id2', 'id4']).reset_index(drop=True)

def _compute_seq_feats(data: pd.DataFrame) -> pd.DataFrame:
    """
    Compute sequential/session features on a pre-sorted (id2, id4) DataFrame.
    Returns a DataFrame indexed by the original df index (via reset_index trick)
    so results can be written back with df.loc[mask, col] = result[col].
    """
    d = data.reset_index()  # preserves original df index as column 'index'

    # 5A. Time since last offer shown to this customer (seconds)
    d['time_since_last_seen'] = (
        d.groupby('id2', observed=True)['id4'].diff().dt.total_seconds().fillna(0)
    )

    # 5B. Cumulative position of this offer within the session (1-indexed)
    d['session_event_count'] = d.groupby('id2', observed=True).cumcount() + 1

    # 5C. Running click count BEFORE this row (NaN y = test rows treated as 0 by cumsum)
    d['no_of_clicks_previously'] = (
        d.groupby('id2', observed=True)['y'].cumsum().shift(1).fillna(0)
    )
    first_rows = d.groupby('id2', observed=True).head(1).index
    d.loc[first_rows, 'no_of_clicks_previously'] = 0

    # 5D. Time since last click (seconds) — forward-fill then shift
    d['_click_ts'] = d.loc[d['y'] == 1, 'id4']
    d['_click_ts'] = d.groupby('id2', observed=True)['_click_ts'].ffill()
    d['_click_ts'] = d.groupby('id2', observed=True)['_click_ts'].shift(1)
    d['time_since_last_click'] = (d['id4'] - d['_click_ts']).dt.total_seconds().fillna(-1)
    d.drop(columns=['_click_ts'], inplace=True)

    # 5E. Previous offer category and subcategory (shifted by 1)
    if 'offer_category' in d.columns:
        d['previous_offer_category'] = (
            d.groupby('id2', observed=True)['offer_category'].shift(1).fillna(-1).astype(int)
        )
    if 'offer_subcategory' in d.columns:
        d['previous_suboffer_category'] = (
            d.groupby('id2', observed=True)['offer_subcategory'].shift(1).fillna(-1).astype(int)
        )

    # 5F. Same-category repeat
    if 'offer_category' in d.columns and 'previous_offer_category' in d.columns:
        d['is_same_category_as_previous'] = (
            (d['offer_category'] == d['previous_offer_category']) &
            (d['previous_offer_category'] != -1)
        ).astype(int)

    return d.set_index('index')  # re-align with original df index


SEQ_FEAT_COLS = [
    'time_since_last_seen', 'session_event_count', 'no_of_clicks_previously',
    'time_since_last_click', 'previous_offer_category', 'previous_suboffer_category',
    'is_same_category_as_previous',
]

train_mask = df['_is_train'] == True
test_mask  = ~train_mask

# ── Train: compute ONLY on train rows ──────────────────────────────────────
# This prevents test rows from inflating session positions / altering time gaps
# for train rows, which was the source of feature leakage.
train_feats = _compute_seq_feats(df[train_mask].copy())

# ── Test: compute on full sequence so test rows see correct train history ───
# We compute on the full df but only extract the test rows' values.
# Train row values are taken from train_feats (above), not from this result.
full_feats = _compute_seq_feats(df.copy())

# ── Write back to df ────────────────────────────────────────────────────────
for col in SEQ_FEAT_COLS:
    if col in train_feats.columns:
        df.loc[train_mask, col] = train_feats[col]
    if col in full_feats.columns:
        df.loc[test_mask, col] = full_feats.loc[test_mask, col]

print('Sequential features (5A–5F) computed without train/test contamination:')
for col in SEQ_FEAT_COLS:
    if col in df.columns:
        print(f'  {col} ✓')

Sequential features (5A–5F) computed without train/test contamination:
  time_since_last_seen ✓
  session_event_count ✓
  no_of_clicks_previously ✓
  time_since_last_click ✓
  previous_offer_category ✓
  previous_suboffer_category ✓
  is_same_category_as_previous ✓


In [103]:
# 5D handled inside _compute_seq_feats above (time_since_last_click)
print('time_since_last_click: handled in sequential features cell ✓')

time_since_last_click: handled in sequential features cell ✓


In [104]:
# 5E handled inside _compute_seq_feats above (previous_offer_category, previous_suboffer_category)
print('previous_offer_category / previous_suboffer_category: handled in sequential features cell ✓')

previous_offer_category / previous_suboffer_category: handled in sequential features cell ✓


In [105]:
# 5F handled inside _compute_seq_feats above (is_same_category_as_previous)
print('is_same_category_as_previous: handled in sequential features cell ✓')

is_same_category_as_previous: handled in sequential features cell ✓


In [106]:
# 5G. Interaction: pace x CTR and session count x CTR
if col_exists(df, 'f366'):
    df['pace_x_ctr']           = df['time_since_last_seen'] * df['f366']
    df['session_count_x_ctr']  = df['session_event_count']  * df['f366']
if col_exists(df, 'f347'):
    df['pace_x_offer_popularity'] = df['time_since_last_seen'] * df['f347']

print('Sequential interaction features created.')

Sequential interaction features created.


---
## 6. Customer-Level Feature Engineering

Aggregates raw customer profile columns (f1–f198) into higher-level signals. All aggregations are row-wise (across feature columns for a single row), not group-by aggregations — so there is no cross-row leakage.

### 6A. Interest Score Aggregations (f1–f12)

f1–f12 encode the customer's declared interest levels across 12 categories (airlines, hotels, dining, etc.).

- `cust_interest_max` — strongest interest signal across all 12 categories
- `cust_interest_mean` — average interest level (breadth of engagement)
- `cust_interest_sum` — total interest mass

In [107]:
interest_cols = existing_cols(df, [f'f{i}' for i in range(1, 13)])
if interest_cols:
    df['cust_interest_max']     = df[interest_cols].max(axis=1)
    df['cust_interest_mean']    = df[interest_cols].mean(axis=1)
    df['cust_interest_std']     = df[interest_cols].std(axis=1).fillna(0)
    df['cust_interest_sum']     = df[interest_cols].sum(axis=1)
    df['cust_interest_nonzero'] = (df[interest_cols] > 0).sum(axis=1)
    df['cust_top_interest']     = safe_idxmax(df, interest_cols).astype('category').cat.codes
    print(f'Interest aggregations created from {len(interest_cols)} columns.')

Interest aggregations created from 10 columns.


### 6B. Digital Engagement

f23–f27 encode the customer's activity across digital channels (e.g. mobile app, web, email).

- `cust_channel_count` — number of active channels (breadth of digital footprint)

Higher values indicate customers who engage across multiple touch points — typically more responsive to offers.

In [108]:
channel_cols = existing_cols(df, ['f23', 'f24', 'f25', 'f26', 'f27'])
if channel_cols:
    df['cust_channel_count'] = df[channel_cols].sum(axis=1)

if col_exists(df, 'f59') and col_exists(df, 'f68'):
    df['cust_time_recency_ratio']         = safe_div(df['f59'].fillna(0), df['f68'].fillna(0))
if col_exists(df, 'f65') and col_exists(df, 'f59'):
    df['cust_travel_page_intensity_30d']  = safe_div(df['f65'].fillna(0), df['f59'].fillna(0))
if col_exists(df, 'f74') and col_exists(df, 'f68'):
    df['cust_travel_page_intensity_180d'] = safe_div(df['f74'].fillna(0), df['f68'].fillna(0))
if col_exists(df, 'f63') and col_exists(df, 'f59'):
    df['cust_mr_page_intensity_30d']      = safe_div(df['f63'].fillna(0), df['f59'].fillna(0))
if col_exists(df, 'f32') and col_exists(df, 'f38'):
    df['cust_email_recency_ratio']        = safe_div(df['f32'].fillna(0), df['f38'].fillna(0))

print('Digital engagement features created.')

Digital engagement features created.


### 6C. Spend Behavior

f152–f162 contain 30-day spend amounts across merchant categories; f163–f173 contain 180-day equivalents.

- `cust_total_spend_30d` — total spend sum across all 30-day category columns
- `cust_total_spend_180d` — total spend sum across all 180-day category columns
- `cust_spend_trend` — ratio `cust_total_spend_30d / cust_total_spend_180d` (recent vs long-term pace; values > 1 mean accelerating spend)

In [109]:
spend_30d_cols  = existing_cols(df, ['f152','f153','f155','f156','f157','f158','f159','f160','f161','f162'])
spend_180d_cols = existing_cols(df, ['f163','f164','f165','f166','f167','f169','f170','f171','f172','f173'])

if spend_30d_cols:
    df['cust_total_spend_30d']   = df[spend_30d_cols].fillna(0).sum(axis=1)
    df['cust_top_spend_cat_30d'] = safe_idxmax(df, spend_30d_cols).astype('category').cat.codes
if spend_180d_cols:
    df['cust_total_spend_180d']  = df[spend_180d_cols].fillna(0).sum(axis=1)
if 'cust_total_spend_30d' in df.columns and 'cust_total_spend_180d' in df.columns:
    df['cust_spend_recency_ratio'] = safe_div(df['cust_total_spend_30d'], df['cust_total_spend_180d'])
if 'cust_total_spend_30d' in df.columns:
    for col, name in [('f159','travel'), ('f157','dining'), ('f158','retail'), ('f153','entmt')]:
        if col_exists(df, col):
            df[f'cust_{name}_spend_share'] = safe_div(df[col].fillna(0), df['cust_total_spend_30d'])
if col_exists(df, 'f39') and col_exists(df, 'f41'):
    df['cust_lifestyle_restaurant_spend'] = df['f39'].fillna(0) + df['f41'].fillna(0)

print('Spend features created.')

Spend features created.


### 6D. Loyalty & Miles

f43 = total loyalty miles accumulated; f45 = number of loyalty segments; f58 = miles expiry flag.

- `cust_miles_per_segment` — average miles per loyalty segment (f43 / f45); high values indicate concentrated loyalty with one program
- Miles expiry flag (f58) is kept as-is — imminent expiry increases redemption motivation

In [110]:
if col_exists(df, 'f43') and col_exists(df, 'f45'):
    df['cust_miles_per_segment']   = safe_div(df['f43'].fillna(0), df['f45'].fillna(0))
if col_exists(df, 'f58'):
    df['cust_loyalty_tenure_yrs']  = df['f58'].fillna(0) / 365.0
if col_exists(df, 'f47') and col_exists(df, 'f43'):
    df['cust_award_miles_ratio']   = safe_div(df['f47'].fillna(0), df['f43'].fillna(0))
if col_exists(df, 'f51') and col_exists(df, 'f58'):
    df['cust_elite_recency_ratio'] = safe_div(df['f51'].fillna(0), df['f58'].fillna(0))

print('Loyalty/miles features created.')

Loyalty/miles features created.


### 6E. Non-Merchant CTR Behavior

f104–f111 contain the customer's historical CTR on non-merchant (editorial/promotional) offer types across 8 categories.

- `cust_best_nonmerchant_ctr` — max CTR across all non-merchant categories (best single-category engagement)
- `cust_avg_nonmerchant_ctr` — mean CTR across categories (general engagement level)

These serve as a baseline "engagement propensity" signal independent of any specific merchant.

In [111]:
nonmerchant_ctr_cols = existing_cols(df, ['f104','f105','f106','f107','f108','f109','f110','f111'])
if nonmerchant_ctr_cols:
    df['cust_best_nonmerchant_ctr'] = df[nonmerchant_ctr_cols].fillna(0).max(axis=1)
    df['cust_mean_nonmerchant_ctr'] = df[nonmerchant_ctr_cols].fillna(0).mean(axis=1)
    df['cust_top_nonmerchant_type'] = safe_idxmax(df, nonmerchant_ctr_cols).astype('category').cat.codes

ctr_trend_cols = existing_cols(df, ['f114', 'f115', 'f118'])
if ctr_trend_cols:
    df['cust_ctr_trend_mean']   = df[ctr_trend_cols].fillna(0).mean(axis=1)
    df['cust_ctr_accelerating'] = (df[ctr_trend_cols].fillna(0) > 1).sum(axis=1)

print('Non-merchant CTR features created.')

Non-merchant CTR features created.


### 6F. Transaction Behavior & Decay Signals

Three sub-groups of features:

**Transaction recency ratio** (f185, f186):
- `cust_txn_recency_ratio` — f185/f186: ratio of recent to total transactions; captures whether customer is currently active

**Transaction category diversity** (f187–f197):
- `cust_txn_diversity` — entropy of spend share across merchant categories; higher = broader spending habits

**Merchant decay CTR** (f147, f148, f149, f150):
- `cust_merchant_decay_ctr_30d` — f147/f148: 30-day decayed CTR on merchant offers
- `cust_merchant_decay_ctr_7d` — f149/f150: 7-day equivalent; captures very recent engagement momentum

In [112]:
# Transaction recency and spend per transaction
if col_exists(df, 'f185') and col_exists(df, 'f186'):
    df['cust_txn_recency_ratio'] = safe_div(df['f185'].fillna(0), df['f186'].fillna(0))
if 'cust_total_spend_30d' in df.columns and col_exists(df, 'f185'):
    df['cust_spend_per_txn_30d'] = safe_div(df['cust_total_spend_30d'], df['f185'].fillna(0))
if 'cust_total_spend_180d' in df.columns and col_exists(df, 'f186'):
    df['cust_spend_per_txn_180d'] = safe_div(df['cust_total_spend_180d'], df['f186'].fillna(0))

In [113]:
# Transaction category diversity
txn_share_cols = existing_cols(df, [f'f{i}' for i in range(187, 198)])
if txn_share_cols:
    df['cust_txn_category_diversity'] = (df[txn_share_cols].fillna(0) > 0).sum(axis=1)
    df['cust_txn_top_category_share'] = df[txn_share_cols].fillna(0).max(axis=1)

In [114]:
# Merchant decay CTR signals
if col_exists(df, 'f147') and col_exists(df, 'f148'):
    df['cust_merchant_decay_ctr_30d']    = safe_div(df['f147'].fillna(0), df['f148'].fillna(0))
if col_exists(df, 'f149') and col_exists(df, 'f146'):
    df['cust_merchant_decay_ctr_14d']    = safe_div(df['f149'].fillna(0), df['f146'].fillna(0))
if col_exists(df, 'f149') and col_exists(df, 'f147'):
    df['cust_decay_click_recency']       = safe_div(df['f149'].fillna(0), df['f147'].fillna(0))
if col_exists(df, 'f201') and col_exists(df, 'f202'):
    df['cust_nonmerchant_decay_ctr_14d'] = safe_div(df['f201'].fillna(0), df['f202'].fillna(0))

print('Transaction & decay features created.')

Transaction & decay features created.


---
## 7. Offer-Level Feature Engineering

Derives signals about the offer's own performance history from the raw offer feature columns (f310–f369).

| Feature | Source columns | Description |
|---|---|---|
| `offer_ctr_momentum_1d_7d` | f310 / f312 | Ratio of 1-day to 7-day offer CTR — positive momentum means trending up |
| `offer_is_high_value` | f217 (min spend threshold) | Binary: 1 if min spend > train median |
| `offer_is_popular` | f222 (impression count) | Binary: 1 if impression count > train 75th percentile |
| `offer_ctr_decay_accel_1d_7d` | f355 − f357 | CTR decay acceleration — how fast the offer is losing engagement |
| `offer_merch_ctr_momentum_1d_7d` | f336 / f338 | Merchant-level 1d/7d CTR ratio (offer's merchant trending up/down) |
| `offer_merch_early_imp_ratio` | f336 / f339 | Fraction of merchant CTR earned in early impressions |
| `offer_popularity` | f314 (overall offer CTR) | Raw overall CTR of the offer |

In [115]:
# CTR Momentum
if col_exists(df, 'f310') and col_exists(df, 'f312'):
    df['offer_ctr_momentum_1d_7d']  = safe_div(df['f310'].fillna(0), df['f312'].fillna(0))
if col_exists(df, 'f311') and col_exists(df, 'f313'):
    df['offer_ctr_momentum_3d_14d'] = safe_div(df['f311'].fillna(0), df['f313'].fillna(0))
if col_exists(df, 'f312') and col_exists(df, 'f314'):
    df['offer_ctr_momentum_7d_30d'] = safe_div(df['f312'].fillna(0), df['f314'].fillna(0))
if col_exists(df, 'f320') and col_exists(df, 'f324'):
    df['offer_imp_momentum_1d_30d'] = safe_div(df['f320'].fillna(0), df['f324'].fillna(0))
if col_exists(df, 'f321') and col_exists(df, 'f324'):
    df['offer_imp_momentum_3d_30d'] = safe_div(df['f321'].fillna(0), df['f324'].fillna(0))
if col_exists(df, 'f315') and col_exists(df, 'f319'):
    df['offer_click_recency_ratio'] = safe_div(df['f315'].fillna(0), df['f319'].fillna(0))
if col_exists(df, 'f314') and col_exists(df, 'f347'):
    df['offer_ctr_vs_avg']          = safe_div(df['f314'].fillna(0), df['f347'].fillna(0))

In [116]:
# Value signals — binary thresholds derived from TRAIN rows only
train_mask = df['_is_train'] == True

if col_exists(df, 'f219') and col_exists(df, 'f217'):
    df['offer_value_per_minspend'] = safe_div(df['f219'].fillna(0), df['f217'].fillna(0))
if col_exists(df, 'f224') and col_exists(df, 'f225'):
    df['offer_time_remaining_pct'] = safe_div(df['f224'].fillna(0), df['f225'].fillna(0))
    df['offer_time_elapsed_pct']   = 1 - df['offer_time_remaining_pct']
if col_exists(df, 'f222'):
    train_median_222 = df.loc[train_mask, 'f222'].median()  # threshold from train only
    df['offer_is_high_value']     = (df['f222'].fillna(0) > train_median_222).astype(int)
if col_exists(df, 'f221'):
    train_median_221 = df.loc[train_mask, 'f221'].median()  # threshold from train only
    df['offer_is_high_mr_points'] = (df['f221'].fillna(0) > train_median_221).astype(int)

In [117]:
# Age & decay
if col_exists(df, 'f355') and col_exists(df, 'f357'):
    df['offer_ctr_decay_accel_1d_7d'] = df['f355'].fillna(0) - df['f357'].fillna(0)
if col_exists(df, 'f355') and col_exists(df, 'f356'):
    df['offer_ctr_decay_accel_1d_2d'] = df['f355'].fillna(0) - df['f356'].fillna(0)
if col_exists(df, 'f358') and col_exists(df, 'f314'):
    df['offer_hyperbolic_ctr_ratio']  = safe_div(df['f358'].fillna(0), df['f314'].fillna(0))

In [118]:
# Merchant CTR momentum
if col_exists(df, 'f336') and col_exists(df, 'f338'):
    df['offer_merch_ctr_momentum_1d_7d']  = safe_div(df['f336'].fillna(0), df['f338'].fillna(0))
if col_exists(df, 'f337') and col_exists(df, 'f340'):
    df['offer_merch_ctr_momentum_3d_30d'] = safe_div(df['f337'].fillna(0), df['f340'].fillna(0))
if col_exists(df, 'f344') and col_exists(df, 'f346'):
    df['offer_merch_early_imp_ratio']     = safe_div(df['f344'].fillna(0), df['f346'].fillna(0))

print('Offer-level features created.')

Offer-level features created.


---
## 8. Customer × Offer Interaction Features

~30 cross features that combine a customer-level signal with an offer-level signal to capture **personalized relevance** — the degree to which this specific customer matches this specific offer.

Key interaction types:

| Feature | Formula | Intuition |
|---|---|---|
| `cx_airline_interest_x_travel_ctr` | f9 × f133 | Customer airline interest × customer travel CTR |
| `cx_dining_ctr_vs_offer` | f130 / f314 | Customer dining CTR relative to offer's overall CTR |
| `cx_dining_spend_x_ctr` | f157 × f130 | Recent dining spend weighted by dining CTR |
| `cx_offer_decay_ctr` | f29 / f28 | Customer's engagement decay on this specific offer |
| `cx_merchant_click_share` | f29 / (f28 + f29) | Customer's share of clicks on this merchant |
| `cx_industry_ctr_vs_offer` | f363 / f314 | Customer industry CTR vs offer benchmark |
| `cx_industry_vs_merchant_ctr` | f363 / f336 | Customer industry CTR vs merchant historical CTR |
| `cx_minspend_feasibility` | `cust_total_spend_30d` / f217 | Can the customer realistically meet the offer's minimum spend? |
| `is_weekend` | f349 ∈ {6,7} | Binary flag for weekend impression |
| `cx_session_depth_x_offer_value` | `session_event_count` × f222 | Deep-in-session impressions of high-value offers |

In [119]:
# Interest x category alignment
if col_exists(df, 'f9')  and col_exists(df, 'f133'):
    df['cx_airline_interest_x_travel_ctr']    = df['f9'].fillna(0)  * df['f133'].fillna(0)
if col_exists(df, 'f8')  and col_exists(df, 'f130'):
    df['cx_restaurant_interest_x_dining_ctr'] = df['f8'].fillna(0)  * df['f130'].fillna(0)
if col_exists(df, 'f11') and col_exists(df, 'f131'):
    df['cx_electronics_interest_x_entmt_ctr'] = df['f11'].fillna(0) * df['f131'].fillna(0)
if col_exists(df, 'f7')  and col_exists(df, 'f132'):
    df['cx_homefurn_interest_x_shopping_ctr'] = df['f7'].fillna(0)  * df['f132'].fillna(0)

In [120]:
# Customer CTR vs offer CTR ratios
if col_exists(df, 'f130') and col_exists(df, 'f314'):
    df['cx_dining_ctr_vs_offer']   = safe_div(df['f130'].fillna(0), df['f314'].fillna(0))
if col_exists(df, 'f133') and col_exists(df, 'f314'):
    df['cx_travel_ctr_vs_offer']   = safe_div(df['f133'].fillna(0), df['f314'].fillna(0))
if col_exists(df, 'f131') and col_exists(df, 'f314'):
    df['cx_entmt_ctr_vs_offer']    = safe_div(df['f131'].fillna(0), df['f314'].fillna(0))
if col_exists(df, 'f132') and col_exists(df, 'f314'):
    df['cx_shopping_ctr_vs_offer'] = safe_div(df['f132'].fillna(0), df['f314'].fillna(0))
if col_exists(df, 'f137') and col_exists(df, 'f314'):
    df['cx_merchant_ctr_vs_offer'] = safe_div(df['f137'].fillna(0), df['f314'].fillna(0))

In [121]:
# Spend x category CTR
if col_exists(df, 'f157') and col_exists(df, 'f130'):
    df['cx_dining_spend_x_ctr']  = df['f157'].fillna(0) * df['f130'].fillna(0)
if col_exists(df, 'f159') and col_exists(df, 'f133'):
    df['cx_travel_spend_x_ctr']  = df['f159'].fillna(0) * df['f133'].fillna(0)
if col_exists(df, 'f158') and col_exists(df, 'f132'):
    df['cx_retail_spend_x_ctr']  = df['f158'].fillna(0) * df['f132'].fillna(0)
if col_exists(df, 'f153') and col_exists(df, 'f131'):
    df['cx_entmt_spend_x_ctr']   = df['f153'].fillna(0) * df['f131'].fillna(0)

In [122]:
# Prior engagement with this offer / merchant
if col_exists(df, 'f29')  and col_exists(df, 'f28'):
    df['cx_offer_decay_ctr']           = safe_div(df['f29'].fillna(0),  df['f28'].fillna(0))
if col_exists(df, 'f31')  and col_exists(df, 'f30'):
    df['cx_merchant_decay_ctr']        = safe_div(df['f31'].fillna(0),  df['f30'].fillna(0))
if col_exists(df, 'f208') and col_exists(df, 'f211'):
    df['cx_merchant_early_click_rate'] = safe_div(df['f208'].fillna(0), df['f211'].fillna(0))
if col_exists(df, 'f209') and col_exists(df, 'f211'):
    df['cx_merchant_7d_click_rate']    = safe_div(df['f209'].fillna(0), df['f211'].fillna(0))
if col_exists(df, 'f205') and 'offer_ctr_momentum_1d_7d' in df.columns:
    df['cx_recent_click_x_momentum']   = df['f205'].fillna(0) * df['offer_ctr_momentum_1d_7d']
if col_exists(df, 'f210') and col_exists(df, 'f346'):
    df['cx_merchant_click_share']      = safe_div(df['f210'].fillna(0), df['f346'].fillna(0))

In [123]:
# Industry CTR alignment (f361-f366)
if col_exists(df, 'f363') and col_exists(df, 'f314'):
    df['cx_industry_ctr_vs_offer']    = safe_div(df['f363'].fillna(0), df['f314'].fillna(0))
if col_exists(df, 'f366') and col_exists(df, 'f314'):
    df['cx_relevant_ctr_vs_offer']    = safe_div(df['f366'].fillna(0), df['f314'].fillna(0))
if col_exists(df, 'f362') and col_exists(df, 'f361'):
    df['cx_industry_click_rate']      = safe_div(df['f362'].fillna(0), df['f361'].fillna(0))
if col_exists(df, 'f365') and col_exists(df, 'f364'):
    df['cx_relevant_click_rate_6mo']  = safe_div(df['f365'].fillna(0), df['f364'].fillna(0))
if col_exists(df, 'f363') and col_exists(df, 'f137'):
    df['cx_industry_vs_merchant_ctr'] = safe_div(df['f363'].fillna(0), df['f137'].fillna(0))

In [124]:
# Offer value vs customer capacity
if col_exists(df, 'f217') and 'cust_total_spend_30d' in df.columns:
    df['cx_minspend_feasibility'] = safe_div(df['cust_total_spend_30d'], df['f217'].fillna(0))
    df['cx_can_meet_minspend']    = (df['cx_minspend_feasibility'] >= 1).astype(int)
if col_exists(df, 'f219') and 'cust_spend_per_txn_30d' in df.columns:
    df['cx_discount_vs_avg_txn']  = safe_div(df['f219'].fillna(0), df['cust_spend_per_txn_30d'])

In [125]:
# Timing context
if col_exists(df, 'f349'):
    df['is_weekend']        = df['f349'].isin([6, 7]).astype(int)
    df['is_weekday']        = df['f349'].isin([1, 2, 3, 4, 5]).astype(int)
    df['is_midweek']        = df['f349'].isin([2, 3, 4]).astype(int)
if col_exists(df, 'f350'):
    df['hour_of_day']       = (df['f350'].fillna(0) // 3600).astype(int)
    df['is_business_hours'] = ((df['hour_of_day'] >= 9) & (df['hour_of_day'] <= 18)).astype(int)
    df['is_evening']        = ((df['hour_of_day'] >= 18) & (df['hour_of_day'] <= 22)).astype(int)
    df['is_morning']        = ((df['hour_of_day'] >= 6)  & (df['hour_of_day'] < 9)).astype(int)
if 'is_weekend' in df.columns and 'offer_is_high_value' in df.columns:
    df['cx_weekend_x_high_value'] = df['is_weekend'] * df['offer_is_high_value']

In [126]:
# Session x offer interactions
if 'session_event_count' in df.columns and col_exists(df, 'f222'):
    df['cx_session_depth_x_offer_value'] = df['session_event_count'] * df['f222'].fillna(0)
if 'no_of_clicks_previously' in df.columns and col_exists(df, 'f366'):
    df['cx_prior_clicks_x_relevant_ctr'] = df['no_of_clicks_previously'] * df['f366'].fillna(0)
if 'time_since_last_click' in df.columns and col_exists(df, 'f366'):
    has_prior_click = (df['time_since_last_click'] != -1).astype(float)
    df['cx_recency_x_relevant_ctr'] = has_prior_click * df['f366'].fillna(0)
if col_exists(df, 'offer_category') and 'previous_offer_category' in df.columns:
    df['cx_offer_category_repeat'] = df['is_same_category_as_previous'] if 'is_same_category_as_previous' in df.columns else 0

print('Customer x Offer interaction features created.')

Customer x Offer interaction features created.


---
## 9. Composite Relevance Score

Combines the 4 highest-signal features into a single weighted score `composite_relevance`:

| Feature | Weight | Correlation with y |
|---|---|---|
| f366 (customer 6-month CTR on relevant offers) | 0.40 | 0.56 |
| f363 (customer CTR on offer's industry, 180d) | 0.30 | 0.50 |
| f365 (customer CTR on offer's category, 180d) | 0.20 | 0.43 |
| f362 (customer CTR on offer's subcategory) | 0.10 | 0.32 |

Weights proportional to correlation with target. Null values filled with 0 before weighting. This score is available to the model both as a standalone feature and as an input to other interaction features.

In [127]:
score_components = {
    'f366': 0.40,   # customer 6mo CTR on relevant offers   (corr=0.56)
    'f363': 0.30,   # customer CTR on offer's industry 180d (corr=0.50)
    'f137': 0.20,   # customer merchant CTR 60d             (corr=0.35)
    'f138': 0.10,   # customer merchant CTR 30d             (corr=0.34)
}
available = {k: v for k, v in score_components.items() if col_exists(df, k)}

if available:
    total_w = sum(available.values())
    df['cx_relevance_score'] = sum(
        df[col].fillna(0) * (w / total_w) for col, w in available.items()
    )
    raw_corr = df['cx_relevance_score'].corr(df['y'])
    print(f'cx_relevance_score created from: {list(available.keys())}')
    print(f'  Raw correlation with y: {raw_corr:.4f}')
else:
    print('No components available.')

cx_relevance_score created from: ['f366', 'f363', 'f137', 'f138']
  Raw correlation with y: 0.4607


---
## 10. Within-Group Relative Rank Features

**Status: REMOVED — data leakage confirmed.**

Originally, percentile rank and z-score features were computed per customer group (e.g. `composite_relevance_grp_pct` = percentile rank of composite score within each customer's full session). These were found to leak future information because:

- `groupby(id2).transform('rank')` uses **all rows in the group**, including future impressions not yet seen at inference time
- At serving time, only past rows are available — the model would have seen a different rank value during training than it would at inference

**Cleanup:** Any pre-existing `_grp_pct` or `_grp_zscore` columns are dropped from `df` before saving, and downstream notebooks (`catboost_lambdamart_ltr_base_model.ipynb`, `lightgbm_lamdamart_ltr_base_model.ipynb`) also drop these columns after loading.

In [128]:
# Within-group rank/zscore features REMOVED — data leakage confirmed.
# These features used full-group statistics (rank over entire customer session,
# groupby transform mean/std) which leak future information into past rows.
# rank_feats block intentionally disabled.
rank_feats = []
new_rank_cols = []
print('Within-group rank/zscore features disabled (leakage fix).')

Within-group rank/zscore features disabled (leakage fix).


In [129]:
# Drop any pre-existing _grp_pct / _grp_zscore columns if present (cleanup)
leaky_cols = [c for c in df.columns if c.endswith('_grp_pct') or c.endswith('_grp_zscore')]
if leaky_cols:
    df.drop(columns=leaky_cols, inplace=True)
    print(f'Dropped {len(leaky_cols)} leaky within-group rank/zscore columns.')
else:
    print('No leaky rank/zscore columns found.')
new_rank_cols = []

No leaky rank/zscore columns found.


In [130]:
# Correlation analysis for group-rank features skipped — features removed (leakage fix).
print('Group-rank correlation analysis skipped (features removed).')

Group-rank correlation analysis skipped (features removed).


---
## 11. Summary

Validates the output dataset:
1. Lists total column count and how many are original f-columns vs engineered features
2. Counts how many OHE columns were dropped
3. Checks that all engineered feature columns have zero nulls
4. Prints the top 20 engineered features by absolute Pearson correlation with `y`

In [131]:
original_cols = set(
    ['id1', 'id2', 'id3', 'id4', 'id5', 'y'] +
    [f'f{i}' for i in range(1, 370)]
)
new_features    = [c for c in df.columns if c not in original_cols]
null_flag_feats = [c for c in new_features if c.endswith('_is_null')]
grp_rank_feats  = []  # removed — leaky features
seq_feats       = [c for c in new_features if c in [
    'time_since_last_seen', 'session_event_count', 'no_of_clicks_previously',
    'time_since_last_click', 'previous_offer_category', 'previous_suboffer_category',
    'is_same_category_as_previous', 'num_offer_categories', 'num_sub_categories',
    'pace_x_ctr', 'session_count_x_ctr', 'pace_x_offer_popularity'
]]
other_feats = [c for c in new_features if c not in null_flag_feats + grp_rank_feats + seq_feats]

In [132]:
print(f'Total columns in featured dataset  : {df.shape[1]}')
print(f'Original features retained         : {len([c for c in df.columns if c in original_cols])}')
print(f'OHE columns dropped                : {len(ohe_cols_to_drop)}')
print()
print(f'New engineered features            : {len(new_features)}')
print(f'  Within-group ranks (REMOVED)     : 0  [leakage fix]')
print(f'  Null indicator flags             : {len(null_flag_feats)}')
print(f'  Composite relevance score        : 1')
print(f'  Sequential features              : {len(seq_feats)}')
print(f'  Other engineered features        : {len(other_feats)}')

Total columns in featured dataset  : 351
Original features retained         : 232
OHE columns dropped                : 84

New engineered features            : 119
  Within-group ranks (REMOVED)     : 0  [leakage fix]
  Null indicator flags             : 12
  Composite relevance score        : 1
  Sequential features              : 12
  Other engineered features        : 95


In [133]:
# Verify no nulls in engineered features
null_check = df[new_features].isnull().mean().sort_values(ascending=False)
has_nulls  = null_check[null_check > 0]

if len(has_nulls):
    print('Features with remaining nulls:')
    print(has_nulls.to_string())
else:
    print('All engineered features: zero nulls ✓')

Features with remaining nulls:
pace_x_offer_popularity   0.2558
cust_interest_mean        0.1167
cust_interest_max         0.1167


In [134]:
print('Top 20 new features by |corr| with y:')
new_corrs = {c: abs(df[c].corr(df['y'])) for c in new_features if df[c].dtype != object}
for k, v in sorted(new_corrs.items(), key=lambda x: -x[1])[:20]:
    print(f'  {k}: {v:.4f}')

Top 20 new features by |corr| with y:
  cx_recency_x_relevant_ctr: 0.5324
  cx_relevant_click_rate_6mo: 0.4890
  cx_relevance_score: 0.4607
  cx_prior_clicks_x_relevant_ctr: 0.4248
  cx_industry_click_rate: 0.4248
  no_of_clicks_previously: 0.4152
  session_count_x_ctr: 0.2358
  cx_restaurant_interest_x_dining_ctr: 0.2268
  cx_homefurn_interest_x_shopping_ctr: 0.2094
  offer_ctr_vs_avg: 0.1615
  cx_airline_interest_x_travel_ctr: 0.1575
  cx_electronics_interest_x_entmt_ctr: 0.1476
  offer_ctr_momentum_1d_7d: 0.1314
  cx_relevant_ctr_vs_offer: 0.1184
  offer_ctr_momentum_7d_30d: 0.0992
  offer_time_remaining_pct: 0.0987
  offer_time_elapsed_pct: 0.0987
  cx_industry_ctr_vs_offer: 0.0946
  offer_ctr_momentum_3d_14d: 0.0853
  cx_merchant_early_click_rate: 0.0776


---
## 12. Save Featured Dataset

Splits the combined `df` back into train and test using the `_is_train` flag:

- **`featured_full_train.csv`** — train rows, `_is_train` column dropped, `y` column retained as target
- **`featured_full_test.csv`** — test rows, `_is_train` column dropped, `y` column retained for final hold-out MAP@7 evaluation

These files are the direct inputs to `catboost_lambdamart_ltr_base_model.ipynb` and `lightgbm_lamdamart_ltr_base_model.ipynb`.

In [135]:
# Split back into train and test
train_featured = df[df['_is_train'] == True].drop(columns=['_is_train'])
# Keep 'y' in test_featured so it can be used as a final hold-out evaluation set (MAP@7)
test_featured = df[df['_is_train'] == False].drop(columns=['_is_train'])

# Save both
train_output = 'featured_full_train.csv'
test_output = 'featured_full_test.csv'

train_featured.to_csv(train_output, index=False)
test_featured.to_csv(test_output, index=False)

print(f'Train featured saved to: {train_output}')
print(f'  Shape: {train_featured.shape}')
print(f'Test featured saved to : {test_output}')
print(f'  Shape: {test_featured.shape}')

Train featured saved to: featured_full_train.csv
  Shape: (616602, 350)
Test featured saved to : featured_full_test.csv
  Shape: (153544, 350)
